Notebook ini ngitung skor tiap fonem, lalu ngebandingin hasilnya sama penilaian ahli. Pilih kernel Python yang dipakai buat install library, terus Run All.

Repo sudah berisi satu WAV asli, model kalibrasi, serta hasil alignment dan posterior Kaldi buat sampel itu. Pengaturan awal langsung pakai file-file tersebut. Ekstraksi audio dan training ulang baru jalan kalau flagnya diaktifkan.

Kaldi menyediakan alignment dan posterior fonem. Gradient boosting belajar memetakan fitur GOP ke skor ahli. Datanya dibagi jadi 2.100 train, 400 validation, dan 2.500 test, dengan penutur yang terpisah. Test ini pernah dipakai di eksperimen sebelumnya.

In [ ]:
from pathlib import Path
import os, sys, subprocess, tempfile
import numpy as np
PROJECT = next(p.resolve() for p in [Path.cwd(), *Path.cwd().parents] if (p / "phoneme_scoring").is_dir())
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
for folder in [".cache/test-temp", ".cache/matplotlib"]:
    (PROJECT / folder).mkdir(parents=True, exist_ok=True)
os.environ.update(OMP_NUM_THREADS="1", OPENBLAS_NUM_THREADS="1", MPLCONFIGDIR=str(PROJECT / ".cache/matplotlib"))
tempfile.tempdir = str(PROJECT / ".cache/test-temp")
RUN_FULL_EXTRACTION = False
RUN_TRAINING = False
RUN_LIVE_INFERENCE = False
RUN_TESTS = False
print("Folder:", PROJECT)
print("Python:", sys.executable)

Audio yang masuk sudah mono 16 kHz dan mengikuti preprocessing handoff. Modul scoring mengecek formatnya, lalu membaca PCM tanpa mengubah gain atau memotong hening. Format yang belum cocok menghasilkan invalid_audio. Waktu tiap fonem dihitung dari awal audio ini.

Kalau mau ulang ekstraksi dan training, taruh wav16k di folder handoff, siapkan runtime lewat tools/bootstrap_native_runtime.py, lalu aktifkan RUN_FULL_EXTRACTION dan RUN_TRAINING. RUN_LIVE_INFERENCE dipakai buat menghitung ulang alignment dan posterior sampel dari audionya.

In [ ]:
from phoneme_scoring.data import read_dataset, dataset_directory
from phoneme_scoring.common import load_json, sha256_file
DATA, CACHE = dataset_directory(PROJECT), PROJECT / "cache"

datasets = read_dataset(DATA, official_counts=False)
for split, rows in datasets.items():
    print(split, len(rows), "rekaman,", len({r["speaker"] for r in rows}), "penutur")
if RUN_FULL_EXTRACTION or RUN_TRAINING:
    missing = [DATA / row["audio_proc"] for rows in datasets.values() for row in rows if not (DATA / row["audio_proc"]).is_file()]
    if missing:
        raise FileNotFoundError(f"Ada {len(missing)} WAV yang belum tersedia. Ekstrak wav16k ke folder manifest sebelum mengulang eksperimen.")

Alignment menentukan potongan audio buat tiap fonem target. Posterior pdf Kaldi digabung ke 39 fonem, lalu log posteriornya dirata-ratakan di sepanjang segmen:

\[
L_i(q)=\frac{1}{T_i}\sum_{t\in A_i}\log P(q\mid x_t),\qquad GOP_i=L_i(p_i)-\max_{q\in Q}L_i(q).
\]

Kalibrator memakai 39 LPP, 39 selisih terhadap target, GOP, log durasi, dan one-hot 39 fonem. Totalnya 119 fitur. Nilai GOP belum langsung sama dengan skor ahli, jadi perlu dipetakan lewat kalibrator.

Alignment awal pakai beam 10 dan retry 40. Rekaman yang masih gagal dicoba lagi dengan beam 100 dan retry 1000. Jalur yang sudah berhasil tetap dipakai. Kalau percobaan kedua juga gagal, skornya null.

In [ ]:
from phoneme_scoring.backend import create_backend
CONFIG = PROJECT / "config/kaldi.native.json"
if RUN_FULL_EXTRACTION:
    subprocess.run([sys.executable, "-m", "phoneme_scoring.batch", "--data", str(DATA), "--cache", str(CACHE), "--kaldi-config", str(CONFIG), "--workers", "4", "--batch-size", "32"], check=True)
stored_metrics = load_json(PROJECT / "reports/gop/metrics.json")
print("Coverage eksperimen yang disimpan:")
for split, coverage in stored_metrics["coverage"].items():
    print(split, coverage["utterances_scored"], "/", coverage["utterances_total"], "rekaman")

Kalibrator dilatih supaya prediksinya mendekati skor ahli. Ada 24 kandidat: enam konfigurasi boosting, masing-masing dicoba dengan empat nilai shrinkage ke median train.

Loss yang dicoba absolute error dan squared error. Jumlah leaf 7/15/31, minimum sampel per leaf 100/50/30, 250 iterasi, learning rate 0,08, dan L2 5. Seed 2026; early stopping dimatikan. Pilihannya ditentukan dari MAE validation. Bobot dari statistik seluruh dataset nggak dipakai buat training.

\[
y_i=\mathrm{skor\ ahli}_i/2,\quad
\hat y_i=(1-\lambda)\operatorname{clip}\left(b+\sum_{m=1}^{250}h_m(z_i),0,1\right)+\lambda\operatorname{median}(y_{train}),\quad S_i=100\hat y_i.
\]

Nilai leaf sudah termasuk learning rate. Pohonnya disimpan di JSON, lalu prediksinya dicocokkan dengan scikit-learn sebelum disimpan. Ambang needs_practice dipilih dari F1 validation dan dikunci sebelum test. Fonem dianggap salah kalau skor ahli < 0,5.

In [ ]:
from phoneme_scoring.experiments import train_experiment
from phoneme_scoring.boosting import load_calibrator
MODEL = PROJECT / "models/calibrator.json"
if RUN_TRAINING:
    assert all((CACHE / f"index_{s}.json").is_file() for s in ["train", "val", "test"]), "Aktifkan RUN_FULL_EXTRACTION untuk membuat cache eksperimen."
    train_experiment(DATA, CACHE, PROJECT / "reports/gop", MODEL)
    subprocess.run([sys.executable, "tools/plot_real_metrics.py"], check=True)
model = load_calibrator(MODEL)
metrics = load_json(PROJECT / "reports/gop/metrics.json")
assert sha256_file(MODEL) == metrics["model_sha256"]
print("Pilihan:", model.artifact["fit_parameters"])
print("Shrinkage:", model.artifact["shrink_to_median"])
print("Ambang latihan (0–1):", model.threshold)
for c in sorted(model.artifact["candidate_validation_results"], key=lambda x:x["validation_mae_raw_0_2"])[:5]:
    print(c)

In [ ]:
for split in ["val", "test"]:
    print("\n", split)
    for name, result in metrics["results"][split].items():
        p = result["phoneme"]; d = p["mispronunciation"]
        print(f"{name:18} N={p['n']:5d} MAE={p['mae_raw_0_2']:.6f} RMSE={p['rmse_raw_0_2']:.6f} Pearson={p['pearson']} F1={d['f1']:.4f} AP={d['pr_auc_average_precision']:.4f}")
print("MAE test melewati baseline:", metrics["beats_baseline_test_mae"])
print("Coverage:", {k:(v["utterances_scored"],v["phones_scored"]) for k,v in metrics["coverage"].items()})

Mayoritas fonem di dataset ini dapat nilai 2. Jadi baseline yang selalu menebak 2 sudah cukup kuat. Selain MAE, lihat juga F1, AP, dan galat pada fonem yang skornya rendah. Korelasi prediksi konstan ditulis null karena nilainya nggak terdefinisi.

Skor kata dan kalimat di sini berasal dari rata-rata skor fonem. Hasilnya tetap dibandingin ke label accuracy ahli, tapi rata-rata itu belum tentu menangkap penilaian keseluruhan. Fluency, intonasi, dan stress belum dinilai.

In [ ]:
from IPython.display import display, Image
display(Image(filename=str(PROJECT / "reports/gop/test_gop_calibrated.png")))
display(Image(filename=str(PROJECT / "reports/gop/test_detection.png")))
r = metrics["results"]["test"]["gop_calibrated"]
print("Kata:", r["word_summary_vs_expert"])
print("Kalimat:", r["sentence_summary_vs_expert"])
print("Fonem skor rendah:", r["low_score_subset"])
for group, value in r["groups"]["age_group"].items():
    print(group, "MAE", value["mae_raw_0_2"], "N", value["n"])

Empat rekaman yang awalnya gagal akhirnya dapat jalur alignment setelah retry. Tapi prediksi skornya masih jauh di atas nilai ahli. Alignment yang berhasil belum berarti penilaiannya sudah tepat. Angka di bawah diambil dari catatan retry pada laporan eksperimen.

In [ ]:
recovery = metrics.get("alignment_recovery", {})
if recovery and recovery.get("model_sha256") != sha256_file(MODEL):
    print("Catatan retry ini berasal dari model eksperimen sebelumnya.")
for row in recovery.get("recordings", []):
    print(row["utt_id"], row["phones"], "fonem; prediksi", round(row["predicted_mean_score_100"], 2), "vs ahli", round(row["expert_mean_score_100"], 2))
if not recovery:
    print("Laporan ini tidak menyertakan catatan retry tersimpan.")

In [ ]:
print("Mode:", "inferensi Kaldi langsung" if RUN_LIVE_INFERENCE else "replay evidence Kaldi dari sampel WAV asli")
from phoneme_scoring import PronunciationScorer
from phoneme_scoring.evidence import CachedBackend
words = load_json(PROJECT / "examples/reference.json")["reference_words"]
backend = create_backend(CONFIG) if RUN_LIVE_INFERENCE else CachedBackend(PROJECT / "examples/evidence", model.extractor_id)
scorer = PronunciationScorer(backend, model)
result = scorer.score_pronunciation(PROJECT / "examples/real_sample.wav", None, words)
assert result["status"] == "ok", result
for word in result["words"]:
    print(word["word"], round(word["phoneme_accuracy_summary"], 2), [(p["target"], round(p["score"], 1), p["needs_practice"]) for p in word["phones"]])
failed = scorer.score_pronunciation(np.zeros(16000), 16000, words)
print("Rekaman hening:", failed["error"])

In [ ]:
if RUN_TESTS:
    test_run = subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"], capture_output=True, text=True)
    print("Tes kontrak output dan pengecekan eksperimen, terpisah dari evaluasi audio.")
    print(test_run.stdout)
    print(test_run.stderr)
    test_run.check_returncode()

Metrik dan grafik ada di `reports/gop`. CSV prediksi dibuat saat training ulang dan nggak ikut repo. Model yang dipakai ada di `models/calibrator.json`.

Perhitungan GOP mengacu ke [implementasi Kaldi](https://kaldi-asr.org/doc/compute-gop_8cc.html). Dataset dan cara penilaian ahlinya dijelaskan di [paper SpeechOcean762](https://www.isca-archive.org/interspeech_2021/zhang21x_interspeech.pdf).